**Drive mount + imports**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
import os
import numpy as np

Mounted at /content/drive


**zip extract**

In [2]:
import zipfile

SOURCE_ZIP = '/content/drive/MyDrive/Thesis /Final /archive.zip'
DEST_PATH = '/content/data'

os.makedirs(DEST_PATH, exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP, 'r') as z:
    z.extractall(DEST_PATH)

print("Done.")

Done.


**DATA_PATH & split**

In [3]:
DATA_PATH = '/content/data/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone'
classes = ['Normal', 'Cyst', 'Tumor', 'Stone']

!pip install split-folders -q
import splitfolders

splitfolders.ratio(DATA_PATH, output="/content/split_data",
                    seed=42, ratio=(.7, .15, .15))

Copying files: 12446 files [00:14, 878.96 files/s] 


**dataset loaders**

In [4]:
IMG_SIZE = (224, 224)
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/split_data/train', image_size=IMG_SIZE, batch_size=BATCH, shuffle=True, seed=42)
val_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/split_data/val', image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)
test_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/split_data/test', image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 8710 files belonging to 4 classes.
Found 1865 files belonging to 4 classes.
Found 1871 files belonging to 4 classes.
Classes: ['Cyst', 'Normal', 'Stone', 'Tumor']


**augmentation + preprocessing**

In [5]:
from tensorflow.keras.applications.convnext import preprocess_input

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomZoom(0.08),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
])

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y)).prefetch(AUTOTUNE)

val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y)).prefetch(AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (preprocess_input(x), y)).prefetch(AUTOTUNE)

**Cell — Model Define**

In [6]:
base_model = tf.keras.applications.ConvNeXtTiny(
    include_top=False, weights='imagenet', input_shape=(224,224,3))
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ convnext_tiny (Functional)      │ (None, 7, 7, 768)      │    27,820,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 768)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         3,076 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,823,204 (106.14 MB)

 Trainable params: 3,076 (12.02 KB)

 Non-trainable params: 27,820,128 (106.13 MB)

**Cell — Training**

In [8]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 161s 535ms/step - accuracy: 0.6448 - loss: 0.9348 - val_accuracy: 0.6681 - val_loss: 0.9166
Epoch 2/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 137s 501ms/step - accuracy: 0.7633 - loss: 0.6578 - val_accuracy: 0.7142 - val_loss: 0.7953
Epoch 3/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 137s 501ms/step - accuracy: 0.8032 - loss: 0.5626 - val_accuracy: 0.7180 - val_loss: 0.7841
Epoch 4/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 145s 513ms/step - accuracy: 0.8202 - loss: 0.5187 - val_accuracy: 0.7340 - val_loss: 0.7487
Epoch 5/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 142s 516ms/step - accuracy: 0.8287 - loss: 0.4823 - val_accuracy: 0.7196 - val_loss: 0.7492
Epoch 6/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 143s 522ms/step - accuracy: 0.8358 - loss: 0.4642 - val_accuracy: 0.7292 - val_loss: 0.7125
Epoch 7/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 147s 536ms/step - accuracy: 0.8444 - loss: 0.4444 - val_accuracy: 0.6971 - val_loss: 0.7489
Epoch 8/10
273/273 ━━━━━━━━━━━━━━━━━━━━ 141s 515ms/step - accuracy: 0.8480 -

**Cell — Evaluate:**

In [9]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

loss, acc = model.evaluate(test_ds)
print(f'Test accuracy: {acc:.4f}, Test loss: {loss:.4f}')

y_true, y_pred = [], []
for x, y in test_ds:
    preds = model.predict(x, verbose=0)
    y_true.extend(y.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))

59/59 ━━━━━━━━━━━━━━━━━━━━ 8s 142ms/step - accuracy: 0.7280 - loss: 0.7047
Test accuracy: 0.7280, Test loss: 0.7047
              precision    recall  f1-score   support

        Cyst       0.79      0.95      0.86       557
      Normal       0.94      0.66      0.78       763
       Stone       0.33      0.74      0.46       208
       Tumor       0.86      0.50      0.64       343

    accuracy                           0.73      1871
   macro avg       0.73      0.71      0.68      1871
weighted avg       0.81      0.73      0.74      1871

[[529   4  23   1]
 [ 15 507 220  21]
 [ 46   3 153   6]
 [ 79  25  66 173]]
